# Ekstrakcija i analiza muzičkih elemenata iz audio signala - tutorijal
### Analiza rezultata detekcije tempa (BPM) i tonaliteta (key) na skupu audio numera


## Uvod

Digitalna obrada zvuka i muzike (engl. MIR) predstavlja disciplinu koja se bavi automatskim izdvajanjem muzičkih i akustičkih karakteristika iz audio signala, korišćenjem metoda digitalne obrade signala i mašinskog učenja *(Downie, 2003)*. Ova oblast leži na preseku nekoliko naučnih disciplina: fizike zvuka (akustike), matematike (Furijeove analize i teorije signala), informatike i muzičke teorije, te predstavlja izuzetno interdisciplinaran teren za istraživanje. U praktičnom smislu, MIR tehnike se svakodnevno koriste u aplikacijama koje veliki broj ljudi koristi bez razmišljanja o pozadinskoj tehnologiji: prepoznavanje pesama (Shazam), automatsko generisanje plejlisti prema raspoloženju ili žanru (Spotify), sinhronizacija svetlosnih efekata sa muzikom, DJ softver koji automatski usklađuje tempo numera, kao i alati za transkripciju akorada i nota za muzičare.

Ovaj rad, koncipiran kao praktični tutorijal u okviru diplomskog rada, bavi se dvema od najfundamentalnijih muzičkih karakteristika koje se mogu automatski izdvojiti iz snimljenog audio signala:

1. **Tempo numere, izražen u BPM-u** (*beats per minute* - broj otkucaja u minuti), koji opisuje brzinu izvođenja i predstavlja osnovu za sinhronizaciju i osećaj ritma u muzici.
2. **Tonalitet numere** (engl. *key*), odnosno skup tonova (durski ili molski) oko kojih je harmonski organizovana kompozicija, a koji određuje „boju“ i emocionalni karakter muzičkog dela.

### Zašto je ovaj problem netrivijalan?

Na prvi pogled, moglo bi se pomisliti da je dovoljno „izbrojati otkucaje“ ili „prepoznati note“ kako bi se ovi parametri odredili, međutim, savremena, produkovana muzika (posebno popularna, pop, rok, hip-hop i elektronska muzika) predstavlja **veoma složeni signal**; u svakom trenutku vremena se preklapaju zvuci više instrumenata (bubnjevi, bas, gitare, klavijature, vokal), svaki sa sopstvenim spektrom harmonija, а bubnjevi dodatno unose **neharmonijske, komponente slične šumu** koje otežavaju analizu tonaliteta. Zbog toga se u ovom radu koristi kombinacija više tehnika:

- **Kratkotrajna Furijeova transformacija - STFT** (*Short-Time Fourier Transform*), kao osnovni alat za prelazak iz vremenskog u frekvencijski domen *(Casey et al, 2008)*;
- **Transformacija sa konstantnim Q faktorom - CQT** (*Constant-Q Transform*), koja frekvencijsku osu predstavlja logaritamski, na način koji znatno bolje odgovara muzičkoj (temperovanoj) skali tonova;
- **Harmonijsko-perkusiona separacija - HPSS** (*Harmonic-Percussive Source Separation*), tehnika koja iz signala izdvaja harmonijsku komponentu (tonove, akorde) odvojeno od perkusivne komponente (udarce bubnjeva);
- **Demucs**, model dubokog učenja za razdvajanje izvora zvuka (*music source separation*), koji iz mešanog signala numere izdvaja pojedinačne instrumente: vokal, bas, bubnjevi i ostali instrumenti, što omogućava da se tonalitet i tempo analiziraju direktno nad „čistijim“, izolovanim signalima.

### Struktura i metodologija ovog tutorijala

Podaci koji se analiziraju u ovom notebook-u nisu sirovi audio zapisi, već **rezultati prethodno sprovedene obrade**, tabelarni (`.csv`) fajlovi u kojima je za svaku od 110 muzičkih numera, raspoređenih u više žanrova (akustik, kantri, fank, džez, latino, kpop, kuća, hip-hop, metal, pank, regi, rnb, rok, tehno, trans...), zabeleženo:

- osnovne informacije o numeri (naziv, žanr, trajanje, *sample rate*);
- tempo (BPM) izračunat na osnovu **cele pesme**, na osnovu **izolovanih bubnjeva** (Demucs) i na osnovu **harmonijske/perkusivne separacije** (HPSS);
- tonalitet izračunat istim skupom pristupa, uz odgovarajuću **meru pouzdanosti (confidence)** za svaku detekciju;
- **ručno anotirane (manuelne) referentne vrednosti** tempa i tonaliteta, koje predstavljaju „tačan odgovor“ (ground truth) prema kome se meri kvalitet automatske detekcije;
- meru **slaganja (agreement)** između automatskih metoda.

Analiza je organizovana u **dve velike celine**, koje odgovaraju dvema iteracijama eksperimenta:

1. **Prvi deo** analizira rezultate dobijene korišćenjem isključivo STFT pristupa za detekciju tonaliteta (dataset `results.csv`). Ovaj deo služi kao polazna, bazna tačka (*baseline*), nad kojom se meri i kvantifikuje doprinos kasnijih poboljšanja.
2. **Drugi deo** analizira rezultate dobijene kada je, pored STFT-a, uveden i CQT pristup za detekciju tonaliteta (dataset `new_results.csv`), čime se direktno upoređuje uticaj izbora transformacije na kvalitet detekcije muzičkih elemenata.

Za svaku od ovih celina, u nastavku rada će biti detaljno objašnjena svaka programska ćelija, šta ona radi, koje pandas/Python konstrukcije koristi, kakav je očekivani rezultat njenog izvršavanja i, što je najvažnije, **šta taj rezultat znači u kontekstu muzičke analize i teorije zvuka**.


---
# 1. Analiza rezultata dobijenih korišćenjem isključivo STFT

U ovom, prvom delu praktičnog rada, analiziraju se rezultati eksperimenta u kome je tonalitet svake numere određivan **isključivo na osnovu kratkotrajne Furijeove transformacije (STFT)**. STFT predstavlja standardni i najrasprostranjeniji način za posmatranje toga kako se frekvencijski sadržaj signala menja tokom vremena: signal se deli na kratke, delimično preklapajuće vremenske segmente (tzv. *frame*-ove), a nad svakim od njih se primenjuje diskretna Furijeova transformacija, čime se dobija spektar (raspodela energije po frekvencijama) za taj kratki vremenski isečak. Nizanjem ovakvih spektara kroz vreme dobija se **spektrogram**, dvodimenzionalna reprezentacija signala (vreme x frekvencija), koja predstavlja osnovu za dalju analizu.

Bitno je odmah naglasiti jedno ograničenje STFT-a koje će biti detaljno razmotreno kasnije u radu; frekvencijska osa dobijena STFT-om je **linearna**, razmak između binova (frekvencijskih „kutija“) je konstantan u herchima. Muzički tonovi, međutim, po prirodi zvučnog opažanja i po fizici muzičkih instrumenata, raspoređeni su **logaritamski** (svako povećanje visine tona za jednu oktavu odgovara duplo većoj frekvenciji). Ova neusklađenost je jedan od glavnih razloga zbog kojih će u drugom delu rada biti uveden CQT pristup.

U nastavku ovog dela biće analizirane dve fundamentalne veličine: **greška i tačnost detekcije tempa (BPM)** i **tačnost detekcije tonaliteta (key)**, i to za tri različita ulazna signala nad kojima su algoritmi primenjeni:

- **Song** - cela, originalna, nerazdvojena numera;
- **Demucs** - izolovani bubnjevi (za tempo) odnosno izolovani harmonijski instrumenti (za tonalitet), dobijeni modelom dubokog učenja Demucs;
- **HPSS** - perkusivna komponenta (za tempo) odnosno harmonijska komponenta (za tonalitet), dobijena klasičnom, signal-processing tehnikom harmonijsko-perkusione separacije.


## 1.1 Učitavanje dataset-a

Prvi korak u svakoj analizi podataka jeste učitavanje prethodno pripremljenih rezultata u radnu strukturu podataka. U ovom slučaju koristi se biblioteka **pandas**, standard u Python ekosistemu za rad sa tabelarnim podacima, koja rezultate učitava u strukturu zvanu `DataFrame`, dvodimenzionalnu tabelu sa imenovanim kolonama, analognu tabeli u relacionoj bazi podataka ili radnom listu u Excel-u, ali sa znatno bogatijim skupom operacija za filtriranje, agregaciju i statističku obradu.

In [1]:
import pandas as pd

Sledeća ćelija učitava fajl `results.csv` - tabelu koja sadrži rezultate prve iteracije eksperimenta (isključivo STFT pristup) i prikazuje prvih pet redova pomoću metode `.head()`, čime se dobija brz uvid u strukturu i sadržaj podataka pre nego što se pristupi dubljoj analizi. Ovo je uobičajena i preporučena praksa u svakoj analizi podataka; pre bilo kakve obrade, potrebno je vizuelno potvrditi da su podaci ispravno učitani i da odgovaraju očekivanom formatu.

In [2]:
df = pd.read_csv('./results.csv')
df.head()

,song_name,genre,duration [s],sample_rate,processing_time_demucs,bpm_song,bpm_drums,bpm_hpss,key_song,confidence_song,key_demucs,confidence_demucs,key_hpss,confidence_hpss,manual_bpm,manual_key,agreement_bpm,agreement_key
0,Eric Clapton - Tears In Heaven.wav,acoustic,274.95,44100,154.23,152.00,152.00,152.00,A Major,0.6694,A Major,0.7211,A Major,0.6648,77,A Major,0.00,3
1,Extreme - More Than Words.wav,acoustic,255.56,44100,137.62,92.29,46.14,92.29,F# Major,0.6938,F# Major,0.3700,F# Major,0.6802,92,F# Major,21.75,3
2,Oasis - Wonderwall.wav,acoustic,279.50,44100,149.61,117.45,117.45,117.45,E Major,0.7251,E Major,0.6343,E Major,0.7387,87,F# Minor,0.00,3
3,The Beatles - Blackbird.wav,acoustic,138.41,44100,82.67,93.96,93.96,93.96,G Minor,0.3990,G Minor,0.3579,G Minor,0.3962,94,G Major,0.00,3
4,Tracy Chapman - Fast Car.wav,acoustic,266.26,44100,178.10,103.36,103.36,103.36,A Minor,0.4734,A Major,0.3703,A Minor,0.4777,102,A Major,0.00,2


Rezultat izvršavanja ove ćelije prikazuje tabelu sa sledećim kolonama, čije značenje je od suštinske važnosti za razumevanje ostatka analize:

| Kolona | Značenje                                                                                                                                                                                     |
|---|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| `song_name` | Naziv audio fajla (numere) nad kojim je vršena analiza.                                                                                                                                      |
| `genre` | Muzički žanr kojem numera pripada (npr. *acoustic*, *rock*, *jazz*...).                                                                                                                      |
| `duration [s]` | Trajanje numere u sekundama.                                                                                                                                                                 |
| `sample_rate` | Učestanost odabiranja (broj uzoraka po sekundi kojim je audio signal digitalizovan; ovde iznosi 44100 Hz).                                                                                   |
| `processing_time_demucs` | Vreme (u sekundama) potrebno modelu Demucs da izvrši razdvajanje izvora zvuka za datu numeru - pokazatelj računske zahtevnosti postupka.                                                     |
| `bpm_song`, `bpm_drums`, `bpm_hpss` | Automatski detektovan tempo (u BPM-u) na osnovu cele pesme, izolovanih bubnjeva (Demucs) i perkusivne komponente (HPSS), redom.                                                              |
| `key_song`, `key_demucs`, `key_hpss` | Automatski detektovan tonalitet na osnovu cele pesme, izolovanih harmonijskih instrumenata (Demucs) i harmonijske komponente (HPSS), redom.                                                  |
| `confidence_song`, `confidence_demucs`, `confidence_hpss` | Mera pouzdanosti (vrednost između 0 i 1) koju algoritam za detekciju tonaliteta pridružuje svojoj proceni; što je vrednost bliža 1, algoritam je „sigurniji“ u tačnost sopstvenog rezultata. |
| `manual_bpm`, `manual_key` | **Referentne (ground truth)** vrednosti tempa i tonaliteta, koje je ručno odredio čovek (npr. muzičar ili istraživač), a koje služe kao merilo tačnosti za sve automatske metode.            |
| `agreement_bpm`, `agreement_key` | Numerička mera slaganja između različitih automatskih metoda (koliko se metode međusobno podudaraju u proceni).                                                                              |

Ova tabela predstavlja polaznu osnovu za sve dalje proračune u ovom delu rada. Svaka sledeća ćelija će, u suštini, izvoditi nove kolone iz ovih osnovnih podataka ili računati agregatne statistike nad njima.

## 1.2 Merenje greške u detekciji tempa (BPM)

Pre izračunavanja bilo kakvih statistika, neophodno je definisati **kako se meri greška** automatski detektovanog tempa u odnosu na ručno anotiranu, referentnu vrednost. Naizgled trivijalno rešenje, prosta apsolutna razlika `|predicted - manual|`, u praksi dovodi do sistematski pogrešnih zaključaka, zbog fenomena poznatog u literaturi kao **half-time** i **double-time** detekcije.

Naime, algoritmi za detekciju tempa često „pogode“ pravi ritmički obrazac, ali pogrešno procene na kom **hijerarhijskom nivou pulsa** se on nalazi; umesto da prepoznaju osnovni otkucaj (npr. četvrtinu), algoritam može detektovati dvostruko brži puls (osminu, *double-time*) ili dvostruko sporiji puls (polovinu, *half-time*). Muzički posmatrano, obe alternative su često podjednako „validne“ interpretacije istog ritma, a razlika je isključivo u tome koji nivo metra čovek subjektivno percipira kao „glavni“ otkucaj. Zbog toga se u ovom radu, prilikom računanja greške, dozvoljava da predviđena vrednost bude tačna ako odgovara originalnoj vrednosti, njenoj duplo većoj ili duplo manjoj varijanti, čime se greška svodi na najmanju od te tri mogućnosti.

In [3]:
def bpm_error(predicted, original):
    candidates = [predicted, predicted * 2, predicted / 2]
    errors = [abs(c - original) for c in candidates]

    return min(errors)

Funkcija `bpm_error` implementira upravo opisanu logiku. Za svaki par (predviđena vrednost `predicted`, referentna vrednost `original`) generišu se tri **kandidata**: sama predviđena vrednost, njena duplo veća vrednost (`predicted * 2`, slučaj kada je algoritam detektovao *half-time* u odnosu na referencu) i njena duplo manja vrednost (`predicted / 2`, slučaj *double-time* detekcije). Za svaki kandidat se računa apsolutna razlika u odnosu na referentnu vrednost, a funkcija na kraju vraća **minimalnu** od dobijenih grešaka pomoću ugrađene Python funkcije `min()`. Ovim pristupom se greška tempa svodi na „najbolje moguće“ tumačenje, čime se izbegava kažnjavanje algoritma za grešku koja je, u suštini, muzički dvosmislena, a ne stvarni promašaj.

In [4]:
df['error_song'] = df.apply(lambda row: bpm_error(row['bpm_song'], row['manual_bpm']), axis=1)
df['error_drums'] = df.apply(lambda row: bpm_error(row['bpm_drums'], row['manual_bpm']), axis=1)
df['error_hpss'] = df.apply(lambda row: bpm_error(row['bpm_hpss'], row['manual_bpm']), axis=1)

U ovoj ćeliji se funkcija `bpm_error` primenjuje na svaki red (numeru) tabele posebno, korišćenjem pandas metode `.apply()` uz `axis=1` (što znači da se funkcija primenjuje red po red, a ne kolona po kolona) i anonimne (`lambda`) funkcije koja iz svakog reda izvlači odgovarajuće kolone. Na ovaj način se kreiraju tri nove kolone u tabeli:

- `error_song` - greška tempa izračunata na osnovu detekcije nad celom pesmom;
- `error_drums` - greška tempa izračunata na osnovu detekcije nad izolovanim bubnjevima (Demucs);
- `error_hpss` - greška tempa izračunata na osnovu detekcije nad perkusijama HPSS komponentom.

Ove tri kolone predstavljaju osnovu za sve naredne statističke proračune vezane za tempo.

## 1.3 Srednja apsolutna greška tempa (BPM MAE)

Kako bismo tri metode uporedili jednim, sažetim brojem po metodi (umesto da posmatramo grešku za svaku od 110 numera pojedinačno), koristi se standardna statistička mera **MAE** (Mean Absolute Error, odnosno srednja apsolutna greška, predstavlja prosečnu apsolutnu razliku između predviđenih i stvarnih vrednosti i često se koristi za evaluaciju regresionih modela *(Willmott & Matsuura, 2005)*). MAE se dobija prostim usrednjavanjem apsolutnih grešaka preko svih numera u skupu podataka, a njena vrednost se izražava u istoj jedinici kao i posmatrana veličina, u ovom slučaju u BPM-u. Što je MAE manja, algoritam je u proseku precizniji; MAE od, recimo, 2 BPM znači da automatska detekcija tempa u proseku odstupa svega 2 otkucaja u minuti od ručno određene referentne vrednosti, što je za praktičnu primenu (npr. sinhronizaciju ili DJ softver) zanemarljivo odstupanje.

In [6]:
mae_song = df['error_song'].mean()
mae_drums = df['error_drums'].mean()
mae_hpss = df['error_hpss'].mean()

print("BPM MAE")
print("=" * 20)
print(f'Song MAE: {round(mae_song, 2)}')
print(f'Drums MAE: {round(mae_drums, 2)}')
print(f'HPSS MAE: {round(mae_hpss, 2)}')

BPM MAE
Song MAE: 5.71
Drums MAE: 4.35
HPSS MAE: 4.48


Kao i što je očekivano, BPM MAE je najmanja za **izolovane bubnjeve korišćenjem Demucs modela**.

Ovaj rezultat je muzikološki intuitivan, bubnjevi predstavljaju perkusivni instrument čiji signal karakterišu jasni, oštri, impulsni udari, koji direktno definišu ritmičku strukturu numere. Kada se ovaj signal izoluje od ostatka mešavine (vokala, harmonijskih instrumenata), algoritam za detekciju otkucaja (koji obično radi tako što traži periodičnost u tzv. *onset* funkciji, funkciji koja meri iznenadne promene energije signala tokom vremena) ima znatno „čistiji“ ulazni signal, bez ometajućih signala drugih instrumenata koji mogu lažno sugerisati pogrešan puls.

Medjutim, i **HPSS metoda** daje vrlo dobar rezultat, dok je BPM MAE **na celoj pesmi** znatno veća u odnosu na prethodne dve metode. HPSS razdvaja perkusivnu komponentu od harmonijske na osnovu razlike u strukturi njihovih spektrograma (perkusivni zvuci se manifestuju kao vertikalne linije u spektrogramu, kratke u vremenu, široke u frekvenciji, dok su harmonijski zvuci horizontalne linije, uske u frekvenciji, dugotrajne u vremenu), pa iako ne postiže potpunu preciznost neuronskog Demucs modela, i dalje znatno poboljšava detekciju tempa u odnosu na analizu cele, nerazdvojene numere, u kojoj se svi instrumenti međusobno „ometaju“.

## 1.4 Tačnost detekcije tempa (BPM Accuracy)

Pored MAE, koja daje uvid u prosečno odstupanje, korisno je posmatrati i **binarnu tačnost** (accuracy), procenat numera za koje je detekcija tempa bila „dovoljno dobra“, prema unapred definisanom pragu tolerancije. Ova mera je intuitivnija za tumačenje jer direktno odgovara na pitanje: „u koliko procenata slučajeva bi korisnik softvera bio zadovoljan rezultatom?“

In [7]:
TOLERANCE = 5

Ovde se definiše konstanta `TOLERANCE`, postavljena na vrednost **5 BPM**. Ovaj prag predstavlja granicu unutar koje se detekcija tempa smatra „ispravnom“, odabran je kao razuman kompromis, dovoljno strog da odbaci grube greške, ali dovoljno tolerantan da uvaži sitna, praktično zanemarljiva odstupanja koja mogu nastati usled male varijacije tempa unutar same numere.

In [8]:
df["song_bpm_correct"] = (df["error_song"] <= TOLERANCE)
df["drums_bpm_correct"] = (df["error_drums"] <= TOLERANCE)
df["hpss_bpm_correct"] = (df["error_hpss"] <= TOLERANCE)

Ova ćelija za svaku od tri metode (`song`, `drums`, `hpss`) kreira novu boolean (logičku) kolonu koja ima vrednost `True` ukoliko je prethodno izračunata greška (`error_song`, `error_drums`, `error_hpss`) manja ili jednaka definisanoj toleranciji od 5 BPM, a `False` u suprotnom. Ove kolone predstavljaju osnovu za direktno računanje procenta tačnosti u sledećoj ćeliji.

In [9]:
song_bpm_acc = (df["song_bpm_correct"].mean() * 100)
drums_bpm_acc = (df["drums_bpm_correct"].mean() * 100)
hpss_bpm_acc = (df["hpss_bpm_correct"].mean() * 100)

print(f"[BPM]: Song Accuracy:  {round(song_bpm_acc, 2)}%")
print(f"[BPM]: Drums Accuracy: {round(drums_bpm_acc, 2)}%")
print(f"[BPM]: HPSS Accuracy:  {round(hpss_bpm_acc, 2)}%")

[BPM]: Song Accuracy:  81.82%
[BPM]: Drums Accuracy: 86.36%
[BPM]: HPSS Accuracy:  86.36%


Konačno, procenat tačnosti (accuracy) za svaku metodu dobija se pozivom `.mean()` nad boolean kolonom. Pandas prilikom usrednjavanja logičke kolone tretira `True` kao 1, a `False` kao 0, pa dobijena srednja vrednost predstavlja upravo **udeo tačno detektovanih numera**, koji se potom množi sa 100 kako bi bio izražen u procentima. Rezultati potvrđuju zaključak iz prethodnog odeljka o MAE vrednosti, a to je da metoda zasnovana na izolovanim bubnjevima (Demucs) postiže najveću tačnost, dok HPSS blisko prati, a detekcija nad celom, nerazdvojenom pesmom zaostaje za obe.

## 1.5 Tačnost detekcije tonaliteta (Key Accuracy)

Nakon analize tempa, prelazimo na analizu druge ključne muzičke karakteristike - **tonaliteta** (key). Za razliku od tempa, koji je jednodimenzionalna, kontinualna veličina (broj otkucaja u minuti), tonalitet je **kategorička, diskretna** veličina: predstavlja se kao kombinacija jednog od 12 tonova hromatske skale (C, C#, D, D#, E, F, F#, G, G#, A, A#, H) i jednog od dva moguća **načina (mode)**, durski (Major) ili molski (Minor), što ukupno daje 24 moguća tonaliteta (npr. „A Major“, „G Minor“ itd.).

Tonalitet numere u velikoj meri određuje njen emocionalni karakter; durski tonaliteti se percipiraju kao svetliji, radosniji, dok molski tonaliteti nose tamniji, melanholičniji karakter, a takođe predstavlja informaciju od praktičnog značaja za muzičare (transkripcija, harmonska analiza) i DJ-eve (harmonijsko mešanje numera, tzv. *harmonic mixing*, gde se numere istog ili srodnog tonaliteta lakše kombinuju bez zvučnog sudara).

Automatska detekcija tonaliteta se tipično zasniva na računanju tzv. **hromagrama** (*chroma vector/chromagram*), reprezentacije koja energiju spektra „savija“ u samo 12 kategorija (po jedna za svaki ton hromatske skale, nezavisno od oktave u kojoj se nalazi), a zatim se dobijeni profil poredi (npr. korelacijom) sa unapred definisanim, teorijski utemeljenim profilima durskih i molskih tonaliteta (najpoznatiji su tzv. **Krumhansl–Schmuckler** profili, dobijeni psihoakustičkim eksperimentima). Tonalitet čiji se teorijski profil najbolje poklapa sa izmerenim hromagramom numere proglašava se za detektovani tonalitet, a stepen tog poklapanja (korelacije) predstavlja upravo pomenutu **meru pouzdanosti (confidence)**.

In [10]:
df["song_key_correct"] = (df["key_song"]==df["manual_key"])
df["demucs_key_correct"] = (df["key_demucs"]==df["manual_key"])
df["hpss_key_correct"] = (df["key_hpss"]==df["manual_key"])

song_key_acc = (df["song_key_correct"].mean() * 100)
demucs_key_acc = (df["demucs_key_correct"].mean() * 100)
hpss_key_acc = (df["hpss_key_correct"].mean() * 100)

print(f"Song Key Accuracy:   {round(song_key_acc, 2)}%")
print(f"Demucs Accuracy:     {round(demucs_key_acc, 2)}%")
print(f"HPSS Accuracy:       {round(hpss_key_acc, 2)}%")

Song Key Accuracy:   47.27%
Demucs Accuracy:     53.64%
HPSS Accuracy:       47.27%


Ova ćelija računa i ispisuje tačnost detekcije tonaliteta za sve tri metode, potpuno analogno postupku primenjenom za tempo u prethodnom odeljku; za svaku metodu se kreira boolean kolona koja poredi detektovani tonalitet (`key_song`, `key_demucs`, `key_hpss`) sa ručno anotiranim, referentnim tonalitetom (`manual_key`), a zatim se usrednjavanjem tih kolona dobija procenat tačnosti.

Za razliku od poređenja BPM vrednosti, ovde **nije potrebna nikakva tolerancija**; tonalitet je kategorička veličina, pa je poređenje jednostavno provera potpune jednakosti dva niza karaktera (stringa), npr. da li je `"A Major"` jednako `"A Major"`.

Detekcija tonaliteta je najbolja na izolovanim instrumentima dobijenim Demucs metodom. Medjutim, sam rezultat od 53.64% nije dobar, ali nije ni loš uzimajući u obzir žanrove sa specifičnim karakteristikama.

Analizom je uočeno da Song i HPSS metoda daju identične rezultate. Ovo je važno zapažanje, jer ono ukazuje na to da HPSS harmonijska komponenta, iako uklanja perkusivni (bubnjarski) doprinos spektru, **ne unosi značajnu dodatnu prednost** u odnosu na analizu cele pesme kada je u pitanju detekcija tonaliteta klasičnim STFT pristupom, verovatno zato što bubnjevi doprinose neharmonijskim, širokopojasnim komponentama spektra koje se delimično „razblažuju“, ali suštinski ne menjaju dominantnu raspodelu tonske energije koju STFT hromagram uočava.

## 1.6 Analiza pouzdanosti (Confidence)

Sledeći korak u analizi jeste ispitivanje **korelacije između pouzdanosti algoritma (confidence) i stvarne tačnosti njegove detekcije**. Ako je mera pouzdanosti dobro kalibrisana, trebalo bi da važi pravilo, što je algoritam „sigurniji“ u svoju procenu tonaliteta (viša vrednost confidence-a), to bi ta procena trebalo statistički češće da bude tačna. Ukoliko se ovakva korelacija ne uoči, to je znak da mera pouzdanosti ne predstavlja pouzdan indikator kvaliteta detekcije, što ima direktne implikacije na dalje korišćenje te mere (npr. u ansambl metodama glasanja koje će biti razmotrene kasnije u radu).

In [11]:
bins = [0, 0.4, 0.6, 0.8, 1.0]

labels = [
    "0-0.4",
    "0.4-0.6",
    "0.6-0.8",
    "0.8-1.0"
]

U ovoj ćeliji se definišu **granice (bins)** i odgovarajuće **oznake (labels)** za grupisanje numera po nivou pouzdanosti Demucs detekcije tonaliteta. Vrednosti pouzdanosti (koje se kreću u opsegu od 0 do 1) dele se u četiri intervala: 0–0.4 (niska pouzdanost), 0.4–0.6, 0.6–0.8 i 0.8–1.0 (visoka pouzdanost). Ovakvo grupisanje kontinualne veličine u diskretne kategorije (tzv. *binning*) omogućava jednostavniju i preglednu agregatnu analizu; umesto da posmatramo pojedinačnu, tačnu vrednost pouzdanosti za svaku od 110 numera, posmatramo prosečnu tačnost detekcije unutar svake od četiri grupe.

In [12]:
df["conf_group"] = pd.cut(df["confidence_demucs"], bins=bins, labels=labels)

Pandas funkcija `pd.cut()` svakoj numeri dodeljuje odgovarajuću kategoriju (grupu pouzdanosti) na osnovu vrednosti u koloni `confidence_demucs`, prema granicama i oznakama definisanim u prethodnoj ćeliji, i rezultat smešta u novu kolonu `conf_group`. Ova funkcija je analogna histogramskom grupisanju, svaka numera „upada“ tačno u jedan od četiri unapred definisana intervala pouzdanosti.

In [13]:
confidence_analysis = (df.groupby("conf_group")["demucs_key_correct"].mean() * 100)
print(confidence_analysis)

conf_group
0-0.4      46.153846
0.4-0.6    50.000000
0.6-0.8    55.357143
0.8-1.0    60.000000
Name: demucs_key_correct, dtype: float64


Konačno, korišćenjem operacije `.groupby("conf_group")`, numere se grupišu prema pripadajućoj kategoriji pouzdanosti, a zatim se za svaku grupu računa prosečna tačnost detekcije tonaliteta (kolona `demucs_key_correct`, pomnožena sa 100 radi izražavanja u procentima). Rezultat ove ćelije omogućava direktnu proveru da li viša pouzdanost zaista prati veću tačnost; ukoliko se pokaže da je prosečna tačnost u grupi „0.8–1.0“ znatno veća nego u grupi „0–0.4“, to potvrđuje da je mera pouzdanosti korisna i informativna; u suprotnom (što će, kao što će naredni odeljci pokazati, delimično biti slučaj), to ukazuje na ograničenja pouzdanosti algoritma za detekciju tonaliteta zasnovanog na STFT-u, koji ume da bude „lažno samouveren“ i kada pogrešno odredi tonalitet.

## 1.7 Tačnost detekcije tonaliteta po žanru

Muzički žanrovi se međusobno razlikuju ne samo po instrumentaciji i tempu, već i po **harmonijskoj strukturi**. Neki žanrovi (npr. pop, rok, kantri) se u velikoj meri oslanjaju na jednostavne harmonske progresije unutar jednog, jasno definisanog tonaliteta, dok drugi žanrovi (npr. džez, fank, soul, R&B) često koriste **proširene akorde** (sedmi, deveti, jedanaesti akordi), **modalne akorde** (pozajmljivanje akorada iz paralelnog tonaliteta) i **česte modulacije** (privremene ili trajne promene tonaliteta unutar iste numere). Ovakve harmonijske karakteristike otežavaju automatsku detekciju tonaliteta, jer hromagram numere u tim slučajevima ne odgovara „čisto“ nijednom od 24 standardna teorijska profila tonaliteta. Analiza tačnosti po žanru omogućava da se ova pretpostavka empirijski proveri.

In [14]:
genre_key = (
    df.groupby("genre")["demucs_key_correct"]
    .mean()
    * 100
).sort_values(ascending=False)

genre_key

genre
folk          100.0
disco          80.0
electronic     80.0
reggae         80.0
rap            80.0
acoustic       60.0
country        60.0
blues          60.0
hiphop         60.0
kpop           60.0
rock           60.0
techno         60.0
metal          60.0
trance         60.0
house          40.0
rnb            40.0
pop            40.0
latin          40.0
punk           20.0
funk           20.0
soul           20.0
jazz            0.0
Name: demucs_key_correct, dtype: float64

Ova ćelija grupiše numere prema koloni `genre`, računa prosečnu tačnost Demucs detekcije tonaliteta unutar svakog žanra (u procentima), a zatim rezultat sortira opadajuće (`sort_values(ascending=False)`), tako da su na vrhu liste žanrovi sa najvišom tačnošću detekcije, a pri dnu žanrovi kod kojih algoritam najčešće greši.

Upravo se ovde može uočiti da se najveće greške dešavaju kod žanrova sa specifičnim harmonijskim karakteristikama, poput džeza, fanka, soul-a i slično, što potvrđuje prethodno izneto očekivanje da harmonijski složeniji žanrovi predstavljaju veći izazov za algoritme zasnovane na poređenju sa jednostavnim, teorijskim profilima durskog/molskog tonaliteta.

## 1.8 Sumarna tabela rezultata prvog dela

Radi preglednosti i lakšeg poređenja svih do sada izračunatih metrika, korisno je sve rezultate objediniti u jednu, jedinstvenu tabelu.

In [15]:
summary = pd.DataFrame({

    "Method": [
        "Song",
        "Demucs",
        "HPSS"
    ],

    "BPM MAE": [
        mae_song,
        mae_drums,
        mae_hpss
    ],

    "BPM Accuracy": [
        song_bpm_acc,
        drums_bpm_acc,
        hpss_bpm_acc
    ],

    "Key Accuracy": [
        song_key_acc,
        demucs_key_acc,
        hpss_key_acc
    ]
})

summary

,Method,BPM MAE,BPM Accuracy,Key Accuracy
0,Song,5.711682,81.818182,47.272727
1,Demucs,4.348727,86.363636,53.636364
2,HPSS,4.479955,86.363636,47.272727


## 1.9 Analiza slaganja i neslaganja metoda u detekciji tonaliteta

Do sada je svaka metoda (Song, Demucs, HPSS) analizirana pojedinačno. Međutim, jednako je važno razumeti **odnos između metoda**, da li greše na istim numerama, ili se njihove greške međusobno dopunjuju? Ukoliko bi se pokazalo da svaka metoda ima svoje „specifično područje kompetencije“ (npr. da Demucs tačno prepoznaje tonalitet numera koje HPSS pogrešno prepoznaje, i obrnuto), to bi otvorilo mogućnost da se kombinovanjem više metoda (npr. glasanjem) postigne bolja ukupna tačnost nego što je postiže bilo koja pojedinačna metoda. S druge strane, ukoliko metode uglavnom greše na istim numerama, kombinovanje neće doneti značajno poboljšanje, jer problem verovatno leži u samoj prirodi tih numera (npr. u njihovoj harmonijskoj složenosti), a ne u konkretnom algoritamskom pristupu.

In [16]:
df["only_song_correct"] = (
    df["song_key_correct"]
    &
    ~df["demucs_key_correct"]
    &
    ~df["hpss_key_correct"]
)

df["only_demucs_correct"] = (
    df["demucs_key_correct"]
    &
    ~df["song_key_correct"]
    &
    ~df["hpss_key_correct"]
)

df["only_hpss_correct"] = (
    df["hpss_key_correct"]
    &
    ~df["song_key_correct"]
    &
    ~df["demucs_key_correct"]
)

print("Samo nad pesmom tačno:", df["only_song_correct"].sum())
print("Samo nad Demucs-om tačno:", df["only_demucs_correct"].sum())
print("Samo nad HPSS tačno:", df["only_hpss_correct"].sum())

Samo nad pesmom tačno: 0
Samo nad Demucs-om tačno: 16
Samo nad HPSS tačno: 0


Ova ćelija ispituje tri međusobno isključiva slučaja korišćenjem logičkog operatora **I** (`&`) i operatora **negacije** (`~`, tilda, koji u pandas-u invertuje boolean vrednosti, odnosno,  pretvara `True` u `False` i obrnuto):

- `only_song_correct` - tačno je **samo** detekcija nad celom pesmom, dok su Demucs i HPSS obe pogrešile;
- `only_demucs_correct` - tačna je **samo** Demucs detekcija;
- `only_hpss_correct` - tačna je **samo** HPSS detekcija.

Ove tri nove logičke kolone omogućavaju da se, brojanjem broja vrednosti `True` (pomoću `.sum()`, gde se `True` tretira kao 1), utvrdi koliko puta je svaka metoda bila **jedina** koja je dala tačan odgovor. 

Ovde potvrdjujemo da Song i HPSS metoda daju indentične rezultate, što se manifestuje kroz to da su brojevi `only_song_correct` i `only_hpss_correct` jednaki nuli (ili vrlo bliski nuli), s obzirom na to da, kao što je ranije uočeno, ove dve metode gotovo uvek daju identičan ishod detekcije, pa jedna od njih retko (ili nikad) može biti tačna dok je druga pogrešna.

## 1.10 Slučajevi potpune saglasnosti i potpunog neslaganja sa referencom

Pored analize „jedinstvenih pobeda“ pojedinačnih metoda, korisno je posmatrati i dva ekstremna slučaja: kada se **sve tri metode slažu** sa ručno anotiranim tonalitetom, i kada **sve tri metode greše**. Prvi slučaj predstavlja „lake“, nedvosmislene numere za koje čak i najjednostavniji pristup (analiza cele pesme) daje tačan rezultat, dok drugi slučaj identifikuje numere koje predstavljaju sistemski izazov za sve razmatrane pristupe, što je posebno vredna informacija za dalje unapređenje sistema.

In [17]:
df["all_correct"] = (
    df["song_key_correct"]
    &
    df["demucs_key_correct"]
    &
    df["hpss_key_correct"]
)

print(f"Sve tri metode tačne: {df["all_correct"].sum()}/{df.shape[0]}")

Sve tri metode tačne: 41/110


Ova ćelija kreira kolonu `all_correct`, koja ima vrednost `True` samo ukoliko su **sve tri** metode (Song, Demucs i HPSS) tačno odredile tonalitet numere (logičko I nad sve tri boolean kolone), a zatim ispisuje ukupan broj takvih numera u odnosu na ukupnu veličinu skupa podataka (izraženo kao razlomak `broj_tačnih/ukupan_broj`, gde se ukupan broj dobija pomoću `df.shape[0]`, broj redova tabele).

In [18]:
df["all_wrong"] = (
    ~df["song_key_correct"]
    &
    ~df["demucs_key_correct"]
    &
    ~df["hpss_key_correct"]
)

print(f"Sve tri metode pogrešile: {df["all_wrong"].sum()}/{df.shape[0]}")

Sve tri metode pogrešile: 41/110


Analogno prethodnoj ćeliji, ovde se kreira kolona `all_wrong`, koja ima vrednost `True` ukoliko su **sve tri** metode pogrešile u detekciji tonaliteta (logičko I nad negiranim boolean kolonama).

Broj pesama gde su sve tri metode pogrešile u detekciji tonaliteta je 41 od 110, što predstavlja veliki broj pesama (oko 35%). Ovaj podatak je izuzetno značajan zaključak prvog dela rada: čak i uz upotrebu Demucs modela za izolaciju harmonijskih instrumenata, u više od trećine slučajeva nijedna od tri razmatrane STFT-zasnovane metode ne uspeva da pravilno odredi tonalitet numere. Ovaj nalaz direktno motiviše uvođenje alternativne transformacije (CQT) u drugom delu rada, sa ciljem da se ovaj procenat značajno smanji.

## 1.11 Glasanje između metoda radi poboljšanja tačnosti (Confidence Weighted Voting)

S obzirom na to da svaka pojedinačna metoda ima svoju meru pouzdanosti (confidence) za predloženi tonalitet, prirodna ideja jeste da se **kombinuju predviđanja sve tri metode** u jedinstvenu, potencijalno bolju odluku, putem tzv. **glasanja ponderisanog pouzdanošću** (*confidence weighted voting* - confidence weighted voting predstavlja metodu kombinovanja više modela u kojoj svaki model doprinosi konačnoj odluci proporcionalno svojoj procenjenoj pouzdanosti *(Kuncheva, 2004)*), tehnike koja pripada široj porodici **ansambl metoda** (*ensemble methods* - ensemble methods predstavljaju grupu tehnika mašinskog učenja koje kombinuju predviđanja više modela radi povećanja tačnosti i robusnosti sistema *(Dietterich, 2000)*) u mašinskom učenju. Umesto da se svakoj metodi dodeli jednak, „demokratski“ glas, svaki predlog tonaliteta se ponderiše (množi) odgovarajućom vrednošću pouzdanosti te metode za tu konkretnu numeru, a kao konačan odgovor bira se tonalitet sa najvećim zbirom ponderisanih glasova. Ideja je da metoda koja je „sigurnija“ u svoj predlog treba da ima veći uticaj na konačnu odluku od metode koja je neodlučna.

In [19]:
from collections import defaultdict

def confidence_vote(row):

    votes = defaultdict(float)

    votes[row["key_song"]] += row["confidence_song"]
    votes[row["key_demucs"]] += row["confidence_demucs"]
    votes[row["key_hpss"]] += row["confidence_hpss"]

    return max(
        votes,
        key=votes.get
    )

Funkcija `confidence_vote` implementira upravo opisani mehanizam glasanja, za jedan red (jednu numeru) tabele:

1. Kreira se `votes` - rečnik posebne vrste, `defaultdict(float)`, iz standardne Python biblioteke `collections`. Ovakav rečnik ima korisnu osobinu da, ukoliko se pristupi ključu koji još ne postoji, automatski kreira taj ključ sa podrazumevanom (default) vrednošću `0.0`, umesto da izazove grešku, što pojednostavljuje kod, jer nije potrebno unapred proveravati da li ključ već postoji.
2. Za svaku od tri metode (`key_song`, `key_demucs`, `key_hpss`), odgovarajuća pouzdanost (`confidence_song`, `confidence_demucs`, `confidence_hpss`) se **dodaje** (`+=`) na „glas“ predloženog tonaliteta u rečniku `votes`. Ukoliko se dve metode slože oko istog tonaliteta, njihove pouzdanosti se sabiraju, čime taj tonalitet dobija dodatnu prednost u odnosu na usamljeni predlog treće metode.
3. Na kraju, funkcija `max(votes, key=votes.get)` pronalazi ključ (tonalitet) sa **najvećom** akumuliranom vrednošću glasova i vraća ga kao konačan, „izglasani“ tonalitet za tu numeru.

In [20]:
df["key_confidence_vote"] = df.apply(
    confidence_vote,
    axis=1
)

Ova ćelija primenjuje funkciju `confidence_vote` na svaki red tabele (ponovo korišćenjem `.apply(..., axis=1)`) i rezultat, izglasani tonalitet za svaku numeru, smešta u novu kolonu `key_confidence_vote`.

In [21]:
df["confidence_vote_correct"] = (
    df["key_confidence_vote"]
    ==
    df["manual_key"]
)

accuracy = (
    df["confidence_vote_correct"]
    .mean()
    * 100
)

print(f"Accuracy metode glasanja: {accuracy}%")

Accuracy metode glasanja: 50.0%


Ovde se, analogno prethodnim proračunima tačnosti, kreira boolean kolona `confidence_vote_correct` koja poredi izglasani tonalitet sa ručno anotiranim referentnim tonalitetom, a zatim se izračunava ukupna tačnost metode glasanja (u procentima).

Broj od 50% za metodu glasanja nije čudan, upravo iz razloga što Song i HPSS metoda daju identične rezultate i tako "nadglašavaju" Demucs model. Drugim rečima, budući da Song i HPSS gotovo uvek predlažu isti tonalitet, njihovi glasovi se efektivno udružuju protiv Demucs predloga, čak i kada je upravo Demucs predlog (koji je, kao što je ranije pokazano, statistički najtačniji) onaj koji je zapravo ispravan. Medjutim, ni confidence nije validna mera za ovu metodu, jer je uočeno da model iako pogreši, daje veliki confidence, što znači da se sam mehanizam glasanja ne može osloniti isključivo na vrednost pouzdanosti kao pokazatelj tačnosti, s obzirom na to da algoritam ume da bude „lažno samopouzdan“.

In [22]:
key_comparison = pd.DataFrame({

    "Method": [
        "Song",
        "Demucs",
        "HPSS",
        "Confidence weighted voting"
    ],

    "Accuracy": [

        df["song_key_correct"]
        .mean() * 100,

        df["demucs_key_correct"]
        .mean() * 100,

        df["hpss_key_correct"]
        .mean() * 100,

        df["confidence_vote_correct"]
        .mean() * 100
    ]
})

key_comparison

,Method,Accuracy
0,Song,47.272727
1,Demucs,53.636364
2,HPSS,47.272727
3,Confidence weighted voting,50.000000


Ova poslednja tabela u analizi glasanja objedinjuje tačnost sve četiri pristupa, Song, Demucs, HPSS i Confidence weighted voting, u jednu preglednu strukturu, čime se jasno vidi da uvedena metoda glasanja, iz prethodno objašnjenih razloga, **ne donosi poboljšanje** u odnosu na samostalnu Demucs metodu, već je, naprotiv, blizu proseka Song/HPSS rezultata usled efekta „nadglašavanja“.

## 1.12 Detaljan pregled numera kod kojih sve tri metode greše

Na kraju prvog dela analize, korisno je konkretno pogledati **koje numere** spadaju u onih 41 slučaj gde su sve tri STFT-zasnovane metode pogrešile detekciju tonaliteta, kao i da li postoji obrazac u pogledu žanrovske pripadnosti tih numera, čime se zatvara krug analize i priprema teren za poređenje sa poboljšanom (CQT) metodom u drugom delu rada.

In [23]:
all_wrong_df = df[
    df["all_wrong"]
]

all_wrong_df[
    [
        "genre",
        "song_name",
        "manual_key",
        "key_song",
        "key_demucs",
        "key_hpss"
    ]
]

,genre,song_name,manual_key,key_song,key_demucs,key_hpss
2,acoustic,Oasis - Wonderwall.wav,F# Minor,E Major,E Major,E Major
3,acoustic,The Beatles - Blackbird.wav,G Major,G Minor,G Minor,G Minor
5,blues,Albert King - Born Under A Bad Sign.wav,C# Minor,C# Major,G# Minor,C# Major
7,blues,Robert Johnson - Cross Road Blues.wav,B Major,A# Minor,B Minor,B Minor
11,country,Dolly Parton - Jolene.wav,C# Minor,C# Major,G# Minor,C# Major
12,country,John Denver - Take Me Home Country Roads.wav,A Major,E Major,E Major,E Major
16,disco,Bee Gees - Stayin Alive.wav,F Minor,F Major,F Major,F Major
23,electronic,deadmau5 - Strobe.wav,G# Minor,B Major,B Major,B Major
30,funk,Average White Band - Pick up the Pieces.wav,F Minor,D# Major,C Minor,D# Major
32,funk,Parliament - Give Up The Funk Tear The Roof Of...,E Minor,B Minor,B Minor,B Minor


Ova ćelija prvo filtrira originalnu tabelu `df` koristeći ranije izračunatu logičku kolonu `all_wrong`, čime se dobija podskup `all_wrong_df` koji sadrži samo one numere za koje su sve tri metode pogrešile. Zatim se iz tog podskupa selektuje samo relevantan podskup kolona (`genre`, `song_name`, `manual_key`, `key_song`, `key_demucs`, `key_hpss`) radi preglednijeg prikaza, bez nepotrebnih kolona poput trajanja ili vremena obrade, koje nisu relevantne za ovu specifičnu proveru. Rezultujuća tabela omogućava direktan uvid u to koji je tačan (referentni) tonalitet bio, i šta je svaka od tri metode pogrešno predložila umesto njega, što otvara mogućnost daljeg, muzikološkog razmatranja (npr. da li su pogrešni predlozi možda **relativni** tonaliteti tačnog odgovora, koji dele isti skup tonova, poput C-dur i A-mol).

In [24]:
all_wrong_df["genre"].value_counts()

genre
jazz          4
soul          4
funk          3
house         3
punk          3
rnb           3
acoustic      2
blues         2
country       2
hiphop        2
latin         2
techno        2
trance        2
disco         1
electronic    1
kpop          1
metal         1
pop           1
reggae        1
rock          1
Name: count, dtype: int64

Ova ćelija koristi metodu `.value_counts()` nad kolonom `genre` podskupa `all_wrong_df`, čime se za svaki žanr prebrojava koliko numera tog žanra spada u grupu gde su sve tri metode pogrešile, sortirano opadajuće po broju pojavljivanja. Ovaj rezultat potvrđuje i kvantifikuje ranije zapažanje iz odeljka 1.7. Žanrovi poput fanka, džeza, latino muzike i soul-a, koji karakterišu harmonijski složenije i manje „standardne“ harmonske progresije, doprinose većem ukupnom broju pogrešnih detekcija, dok su drugi žanrovi zastupljeni sa svega jednim ili nijednim slučajem greške.

---
### Zaključak prvog dela analize

Analiza rezultata dobijenih isključivo STFT pristupom pokazuje da:

- **Detekcija tempa (BPM)** je, uz razmatranje half-time/double-time fenomena, izuzetno pouzdana, naročito kada se primeni nad izolovanim bubnjevima (Demucs), a HPSS metoda daje uporediv, vrlo dobar rezultat uz znatno manju računsku cenu (bez potrebe za neuronskim modelom).
- **Detekcija tonaliteta** predstavlja znatno teži problem: čak i najbolja STFT zasnovana metoda (Demucs) postiže tačnost od svega oko 53.64%, dok u čak 35% slučajeva sve tri metode zajedno greše.
- Song i HPSS metoda daju gotovo identične rezultate u detekciji tonaliteta, što ukazuje da HPSS separacija, primenjena na klasičan (STFT) hromagram, ne donosi značajnu dodatnu vrednost za ovaj konkretan zadatak.
- Naivno glasanje između metoda, ponderisano merom pouzdanosti, ne poboljšava rezultat, jer (a) Song i HPSS „nadglašavaju“ Demucs usled svoje međusobne sličnosti, i (b) mera pouzdanosti nije pouzdana, algoritam ume da bude samouveren i kada greši.
- Žanrovi sa harmonijski složenijom strukturom (džez, fank, soul, latino) sistematski predstavljaju najveći izazov za sve razmatrane pristupe.

Ovi nalazi jasno motivišu potrebu za drugačijim pristupom analizi tonaliteta, koji bi bolje odgovarao logaritamskoj prirodi muzičke visine tona, što nas dovodi do drugog dela rada, u kome se uvodi **CQT (Constant-Q Transform)**.


---

# 2. Analiza rezultata dobijenih korišćenjem STFT+CQT

**CQT (Constant-Q Transform, transformacija sa konstantnim Q faktorom)** je, za razliku od STFT metode koja odgovara opštoj obradi signala, transformacija koja mnogo bolje odgovara muzici, jer poseduje **logaritamsku frekvencijsku skalu**, za razliku od STFT-a koja ima **linearnu** frekvencijsku skalu *(Müller, 2015)*.

Da bismo razumeli zašto je ova razlika toliko važna, potrebno je prisetiti se fizike muzičke visine tona. U muzičkoj skali, frekvencije nota rastu **geometrijski** (a ne aritmetički), svaki polustepen (najmanji korak u standardnoj muzičkoj skali, npr. razlika između tona C i C#) odgovara množenju frekvencije faktorom $2^{1/12} \approx 1.0595$, dok cela oktava (12 polustepeni) odgovara tačnom udvostručenju frekvencije. To znači da je razmak u hercima između, na primer, tonova C2 i C#2 (niski registar) mnogo manji nego razmak između tonova C6 i C#6 (visoki registar), iako muzički oba para predstavljaju identičan interval, odnosno jedan polustepen.

STFT, sa svojom linearnom frekvencijskom rezolucijom (konstantan broj herca po binu, nezavisno od apsolutne frekvencije), ovu razliku ne uvažava. U niskom registru, gde su note gusto raspoređene u hercima, STFT ima nedovoljnu rezoluciju da ih razdvoji, dok u visokom registru „rasipa“ nepotrebno mnogo binova na razmak koji muzički predstavlja isti interval kao u niskom registru. CQT rešava ovaj problem tako što frekvencijske binove raspoređuje **geometrijski** (svaki sledeći bin je za konstantan množilac veći od prethodnog, a ne za konstantnu razliku u hercima), sa gustinom binova po oktavi koja odgovara broju polustepeni (tipično 12, 24 ili 36 binova po oktavi), čime se rezolucija transformacije direktno usklađuje sa muzičkom, a ne isključivo fizičkom, strukturom zvuka. Zbog ovoga se očekuje da hromagram izračunat na osnovu CQT-a bude precizniji i muzikološki verniji reprezentant tonske sadržine signala, što bi trebalo da se direktno odrazi na tačnost detekcije tonaliteta.

Važno je napomenuti da se u ovom, drugom delu analize, **CQT koristi isključivo za detekciju tonaliteta** (koji zavisi od precizne tonske/frekvencijske informacije), dok se **detekcija tempa i dalje oslanja na STFT** odnosno na *onset* funkciju, budući da za otkrivanje ritmičkih obrazaca nije neophodna fina frekvencijska rezolucija, već pre svega dobra **vremenska** rezolucija, koju STFT već obezbeđuje u zadovoljavajućoj meri.

## 2.1 Učitavanje novog dataset-a

Analogno prvom delu rada, prvi korak je učitavanje rezultata druge iteracije eksperimenta, ovoga puta iz fajla `new_results.csv`, koji pored svih ranije opisanih kolona sadrži i tri dodatne: `key_song_cqt`, `key_demucs_cqt` i `key_hpss_cqt`, tonalitet detektovan CQT pristupom, nad istim signalima (cela pesma, Demucs, HPSS) kao i u STFT slučaju. Ovo omogućava direktno poređenje STFT i CQT pristupa nad identičnim skupom numera.

In [25]:
import pandas as pd
import numpy as np

df = pd.read_csv('./new_results.csv')
df.head()

,song_name,genre,duration[s],sample_rate,processing_time_demucs[s],bpm_song,bpm_drums,bpm_hpss,key_song,key_demucs,key_hpss,key_song_cqt,key_demucs_cqt,key_hpss_cqt,manual_bpm,manual_key,agreement_bpm,agreement_key
0,Eric Clapton - Tears In Heaven.wav,acoustic,274.95,44100,161.59,152.00,152.00,152.00,A Major,A Major,A Major,A Major,A Major,A Major,77,A Major,0.00,3
1,Extreme - More Than Words.wav,acoustic,255.56,44100,146.82,92.29,46.14,92.29,F# Major,F# Major,F# Major,F# Major,F# Major,F# Major,92,F# Major,21.75,3
2,Oasis - Wonderwall.wav,acoustic,279.50,44100,159.65,117.45,117.45,117.45,E Major,E Major,E Major,F# Minor,F# Minor,A Major,87,F# Minor,0.00,3
3,The Beatles - Blackbird.wav,acoustic,138.41,44100,78.89,93.96,93.96,93.96,G Minor,G Minor,G Minor,G Major,G Major,G Major,94,G Major,0.00,3
4,Tracy Chapman - Fast Car.wav,acoustic,266.26,44100,151.47,103.36,103.36,103.36,A Minor,A Major,A Minor,A Major,A Major,A Major,102,A Major,0.00,2


Osim ranije korišćene biblioteke `pandas`, ovde se dodatno uvozi i **NumPy** (`numpy`, alijas `np`), biblioteka za numeričke proračune koja se, kao što će se videti u narednoj ćeliji, koristi radi ispravnog rukovanja **nedostajućim vrednostima** (`np.nan`), koje se u ovom, proširenom skupu podataka mogu javiti u koloni `manual_bpm` (npr. za numere kod kojih ručna anotacija tempa iz nekog razloga nije bila dostupna ili pouzdana). Ćelija zatim učitava novi fajl rezultata i prikazuje prvih pet redova, na potpuno isti način kao u prvom delu rada.

## 2.2 BPM analiza sa podrškom za nedostajuće vrednosti

S obzirom na to da novi skup podataka može sadržati numere bez validne ručne anotacije tempa, funkcija za računanje greške mora biti prilagođena tako da takve slučajeve prepozna i ispravno obradi, umesto da izazove grešku ili proizvede besmislen rezultat.

In [26]:
def bpm_error(predicted, manual):
    if pd.isna(manual):
        return np.nan

    candidates = [predicted, predicted * 2,predicted / 2]

    errors = [abs(c - manual) for c in candidates]

    return min(errors)

Ova ćelija ponovo definiše funkciju `bpm_error`, ovoga puta sa dodatnom mera predostrožnosti: na samom početku funkcije proverava se da li je referentna (ručna) vrednost `manual` nedostajuća, pomoću `pd.isna(manual)`, pandas funkcije koja prepoznaje razne oblike „praznih“ vrednosti (NaN, None i slično). Ukoliko referentna vrednost nedostaje, funkcija odmah vraća `np.nan`, čime se izbegava besmisleno računanje greške u odnosu na nepostojeći podatak (što bi, bez ove provere, dovelo do toga da Python pokuša da oduzme broj od praznog/nedefinisanog objekta i izazove izuzetak, ili, u zavisnosti od implementacije, tiho proizvede pogrešan rezultat). Ukoliko referentna vrednost postoji, logika je identična onoj iz prvog dela rada; generišu se tri kandidata (originalna, duplo veća i duplo manja vrednost predviđanja) i vraća se najmanja apsolutna greška među njima.

In [27]:
df["bpm_error_song"] = df.apply(
    lambda row: bpm_error(row["bpm_song"], row["manual_bpm"]), axis=1)

df["bpm_error_drums"] = df.apply(
    lambda row: bpm_error(row["bpm_drums"], row["manual_bpm"]), axis=1)

df["bpm_error_hpss"] = df.apply(lambda row: bpm_error(row["bpm_hpss"], row["manual_bpm"]), axis=1)

Analogno prvom delu rada, ova ćelija primenjuje (novu, NaN-svesnu) funkciju `bpm_error` na sve tri metode (`bpm_song`, `bpm_drums`, `bpm_hpss`) u odnosu na `manual_bpm`, kreirajući tri nove kolone: `bpm_error_song`, `bpm_error_drums` i `bpm_error_hpss`. Numere kod kojih `manual_bpm` nedostaje automatski će u ovim kolonama imati vrednost `NaN`, koja se, kao što ćemo videti u narednim proračunima, automatski izostavlja iz agregatnih statistika poput `.mean()`. Pandas po podrazumevanom podešavanju ignoriše NaN vrednosti prilikom računanja srednje vrednosti, tako da nedostajući podaci ne narušavaju konačan rezultat.

## 2.3 BPM MAE (drugi skup podataka)

Ponovnim računanjem MAE nad novim skupom podataka proveravamo da li je tempo i dalje pouzdano detektovan (očekivano je da rezultati budu slični prvom delu rada, s obzirom na to da metoda detekcije tempa nije menjana, samo je dodata podrška za nedostajuće vrednosti).

In [28]:
print("BPM MAE")
print("=" * 20)
print(f"Song: {round(df["bpm_error_song"].mean(), 2)}")
print(f"Drums: {round(df["bpm_error_drums"].mean(), 2)}")
print(f"HPSS: {round(df["bpm_error_hpss"].mean(), 2)}")

BPM MAE
Song: 5.71
Drums: 4.34
HPSS: 4.48


Ova ćelija ispisuje MAE za sve tri metode (Song, Drums, HPSS) nad novim skupom podataka, formatirano na potpuno isti način kao u prvom delu rada. Ovi rezultati služe prevashodno kao **kontrolna provera konzistentnosti**, budući da se algoritam za detekciju tempa nije menjao između dve iteracije eksperimenta, očekuje se da MAE vrednosti ostanu u istom opsegu kao i u prvom delu rada, čime se potvrđuje da uvođenje CQT-a (koje utiče isključivo na detekciju tonaliteta) nije slučajno unelo neku regresiju ili grešku u deo koda koji se bavi tempom.

## 2.4 BPM Accuracy (±5 BPM)

Analogno prvom delu rada, tačnost detekcije tempa se ponovo izražava kao procenat numera čija greška ne prelazi unapred definisanu toleranciju od 5 BPM.

In [29]:
def bpm_accuracy(error_series):
    return ((error_series <= 5).mean() * 100)

print("BPM Accuracy")
print("=" * 20)
print(f"Song: {round(bpm_accuracy(df["bpm_error_song"]), 2)}%")
print(f"Drums: {round(bpm_accuracy(df["bpm_error_drums"]), 2)}%")
print(f"HPSS: {round(bpm_accuracy(df["bpm_error_hpss"]), 2)}%")

BPM Accuracy
Song: 81.82%
Drums: 86.36%
HPSS: 86.36%


## 2.5 BPM MAE po žanru

Kao dodatak globalnoj (agregatnoj) analizi tempa, korisno je ispitati da li postoje žanrovi kod kojih je detekcija tempa sistematski lošija, na primer, žanrovi sa nepravilnim ritmom (fank, džez) ili žanrovi bez jasno definisanog, konstantnog pulsa (pojedini ambijentalni ili akustični komadi) mogu predstavljati veći izazov od žanrova sa mehanički preciznim, elektronski generisanim ritmom (tehno, haus).

In [30]:
genre_bpm_mae = (
    df.groupby("genre")[["bpm_error_song", "bpm_error_drums", "bpm_error_hpss"]]
      .mean()
      .round(2)
      .rename(columns={
          "bpm_error_song": "Song",
          "bpm_error_drums": "Drums",
          "bpm_error_hpss": "HPSS"
      })
      .sort_index()
)

print("BPM MAE po žanrovima")
genre_bpm_mae

BPM MAE po žanrovima


,Song,Drums,HPSS
genre,,,
acoustic,6.19,6.19,6.19
blues,2.29,6.64,2.29
country,1.79,1.25,1.79
disco,0.87,0.87,0.87
electronic,0.94,0.94,0.94
folk,7.30,11.34,7.90
funk,0.89,0.89,0.89
hiphop,5.02,0.97,0.95
house,0.11,0.11,0.11


Ova ćelija grupiše numere po koloni `genre`, računa prosečnu (`mean()`) grešku tempa za sve tri metode istovremeno (selekcijom liste kolona `[["bpm_error_song", "bpm_error_drums", "bpm_error_hpss"]]` pre poziva `.groupby()`), zaokružuje rezultate na dve decimale radi preglednosti (`.round(2)`), preimenuje kolone u čitljivije nazive (Song, Drums, HPSS) pomoću `.rename(columns=...)`, i na kraju sortira tabelu po abecednom redu naziva žanra (`.sort_index()`).

## 2.6 Analiza tonaliteta - poređenje STFT i CQT pristupa

Dolazimo do centralnog dela ovog poglavlja i, u širem smislu, centralnog doprinosa čitavog rada, direktnog, kvantitativnog poređenja tačnosti detekcije tonaliteta korišćenjem klasičnog STFT pristupa naspram novouvedenog CQT pristupa, nad identičnim skupom numera i identičnim metodama izolacije signala (cela pesma, Demucs, HPSS).

In [31]:
def key_accuracy(column):
    return (df[column] == df["manual_key"]).mean() * 100

Definiše se pomoćna funkcija `key_accuracy(column)`, koja prima naziv kolone (npr. `"key_song"` ili `"key_song_cqt"`) i vraća procenat podudaranja te kolone sa referentnom kolonom `manual_key`. Ova funkcija uvodi generičku, parametrizovanu logiku koja se u narednim ćelijama poziva šest puta (za tri metode x dve transformacije).

#### STFT Accuracy

In [32]:
print("STFT")
print("=" * 20)
print(f"Song: {round(key_accuracy("key_song"), 2)}%")
print(f"Demucs: {round(key_accuracy("key_demucs"), 2)}%")
print(f"HPSS: {round(key_accuracy("key_hpss"), 2)}%")

STFT
Song: 47.27%
Demucs: 53.64%
HPSS: 47.27%


Ova ćelija poziva funkciju `key_accuracy` za tri STFT kolone (`key_song`, `key_demucs`, `key_hpss`) i ispisuje dobijene procente tačnosti. Ovi rezultati treba, u principu, da budu identični (ili vrlo bliski) onima dobijenim u prvom delu rada (odeljak 1.5), budući da je reč o istom algoritmu (STFT) primenjenom nad istim (ili gotovo istim) skupom numera. Ova ćelija stoga služi i kao dodatna potvrda konzistentnosti između dva eksperimenta, pre nego što se pređe na glavno poređenje sa CQT pristupom.

#### CQT Accuracy

In [33]:
print("CQT")
print("=" * 20)
print(f"Song: {round(key_accuracy("key_song_cqt"), 2)}%")
print(f"Demucs: {round(key_accuracy("key_demucs_cqt"), 2)}%")
print(f"HPSS: {round(key_accuracy("key_hpss_cqt"), 2)}%")

CQT
Song: 55.45%
Demucs: 63.64%
HPSS: 53.64%


Analogno prethodnoj ćeliji, ovde se funkcija `key_accuracy` poziva nad tri **CQT** kolone (`key_song_cqt`, `key_demucs_cqt`, `key_hpss_cqt`).

Ovde uočavamo znatno poboljšanje detekcije tonaliteta, gde se korišćenjem CQT i Demucs modela dobija accuracy od 63.64%. U poređenju sa 53.64% koliko je STFT+Demucs kombinacija postizala u prvom delu rada, ovo predstavlja apsolutno poboljšanje od približno 10 procentnih poena, što potvrđuje početnu hipotezu ovog dela rada, a to je da logaritamska frekvencijska skala CQT-a, usklađena sa geometrijskom prirodom muzičke visine tona, omogućava precizniju detekciju tonaliteta u odnosu na linearnu skalu STFT-a.

## 2.7 Poređenje svih šest metoda

Radi celovitog i preglednog uvida, korisno je sve razmatrane kombinacije (tri metode izolacije signala x dve transformacije) rangirati zajedno, u jednoj, sortiranoj tabeli.

In [34]:
methods = [
    "key_song",
    "key_demucs",
    "key_hpss",

    "key_song_cqt",
    "key_demucs_cqt",
    "key_hpss_cqt"
]

In [35]:
results = []

for method in methods:
    accuracy = ((df[method] == df["manual_key"]).mean() * 100)
    results.append([method, accuracy])

accuracy_df = pd.DataFrame(
    results,
    columns=[
        "Method",
        "Accuracy"
    ]
)

accuracy_df.sort_values("Accuracy", ascending=False)

,Method,Accuracy
4,key_demucs_cqt,63.636364
3,key_song_cqt,55.454545
5,key_hpss_cqt,53.636364
1,key_demucs,53.636364
0,key_song,47.272727
2,key_hpss,47.272727


Ne samo da je CQT+Demucs bolji od STFT+Demucs, već je svaka metoda bolja u kombinaciji sa CQT nego sa STFT. Ovaj nalaz je izuzetno važan jer pokazuje da poboljšanje nije ograničeno samo na kombinaciju sa Demucs separacijom, već predstavlja **konzistentnu, opštu prednost** CQT transformacije nad STFT transformacijom, nezavisno od toga da li se primenjuje na celu pesmu, na Demucs izolovane instrumente ili na HPSS harmonijsku komponentu.

## 2.8 Provera: u kom procentu je bar jedna od šest metoda dala tačan rezultat

Prethodne analize su se bavile tačnošću pojedinačnih metoda i njihovim međusobnim poređenjem. Sada posmatramo komplementarno pitanje, a to je koliko je čest slučaj da **bar jedna** od šest raspoloživih metoda (tri STFT + tri CQT) daje tačan tonalitet? Ovo je relevantno jer, ukoliko bi se u budućnosti razvio pouzdaniji mehanizam selekcije „prave“ metode za svaku numeru (npr. bolje kalibrisana mera pouzdanosti ili klasifikator koji na osnovu žanra bira odgovarajuću metodu), gornja granica potencijalnog poboljšanja upravo je određena ovim procentom.

In [36]:
df["any_correct"] = (
    (df["key_song"] == df["manual_key"]) |
    (df["key_demucs"] == df["manual_key"]) |
    (df["key_hpss"] == df["manual_key"]) |
    (df["key_song_cqt"] == df["manual_key"]) |
    (df["key_demucs_cqt"] == df["manual_key"]) |
    (df["key_hpss_cqt"] == df["manual_key"])
)

print(f"Procenat u kom je bar jedna metoda dala tačan rezultat: {round(df["any_correct"].mean() * 100, 2)}%")

Procenat u kom je bar jedna metoda dala tačan rezultat: 83.64%


Ovo predstavlja ogromno poboljšanje u odnosu na ranije rezultate koji su koristili isključivo STFT. Ovim smo pokazali da je u samo 16.36% slučajeva svaka metoda pogrešila tonalitet, što je mnogo bolje od prethodnog rezultata gde je taj procenat bio oko 35%. Ovaj nalaz dodatno naglašava vrednost uvođenja CQT transformacije; ne samo da svaka pojedinačna CQT metoda nadmašuje svoj STFT parnjak, već i **skup svih šest metoda zajedno** pokriva znatno veći deo prostora mogućih numera, otvarajući realnu perspektivu za dalje poboljšanje putem pametnijih, informisanih ansambl tehnika (za razliku od glasanja koje se pokazalo neuspešnim u prvom delu rada).

## 2.9 Kvantifikacija doprinosa CQT metode u odnosu na STFT

Kako bismo precizno kvantifikovali doprinos CQT pristupa, u nastavku se, za svaku od tri metode izolacije signala (cela pesma, Demucs, HPSS) posebno, prebrojava tačan broj numera kod kojih je **CQT dao tačan odgovor, dok je odgovarajuća STFT metoda pogrešila**, što predstavlja direktnu meru „pobede“ CQT-a nad STFT-om, po pojedinačnoj numeri.

#### Za celu pesmu

In [37]:
song_stft = (df["key_song"] == df["manual_key"])
song_cqt = (df["key_song_cqt"] == df["manual_key"])

print(f"CQT je na celoj pesmi pogodio {(song_cqt & ~song_stft).sum()} puta gde je STFT pogrešio.")

CQT je na celoj pesmi pogodio 27 puta gde je STFT pogrešio.


Ova ćelija kreira dve privremene boolean kolone, `song_stft` (da li je STFT detekcija nad celom pesmom tačna) i `song_cqt` (da li je CQT detekcija nad celom pesmom tačna), a zatim, korišćenjem logičkog izraza `song_cqt & ~song_stft` (CQT tačan **i** STFT netačan), prebrojava (`.sum()`) tačan broj numera kod kojih je CQT „spasao“ pogrešnu STFT detekciju, primenjeno na analizu cele, nerazdvojene pesme.

#### Za izolovane instrumente (Demucs)

In [38]:
demucs_stft = (df["key_demucs"] == df["manual_key"])
demucs_cqt = (df["key_demucs_cqt"] == df["manual_key"])

print(f"CQT je na izolovanim Demucs instrumentima pogodio {(demucs_cqt & ~demucs_stft).sum()} puta gde je STFT pogrešio.")

CQT je na izolovanim Demucs instrumentima pogodio 20 puta gde je STFT pogrešio.


Analogna provera, ovoga puta primenjena na Demucs izolovane harmonijske instrumente; poredi se koliko puta je `key_demucs_cqt` bio tačan dok je `key_demucs` (STFT varijanta) bio netačan.

#### Za izolovane instrumente+vokal (HPSS harmonics)

In [39]:
demucs_stft = (df["key_hpss"] == df["manual_key"])
demucs_cqt = (df["key_hpss_cqt"] == df["manual_key"])

print(f"CQT je na izolovanim HPSS harmonics pogodio {(demucs_cqt & ~demucs_stft).sum()} puta gde je STFT pogrešio.")

CQT je na izolovanim HPSS harmonics pogodio 24 puta gde je STFT pogrešio.


Treća, analogna provera se odnosi na HPSS harmonijsku komponentu, poredi se koliko puta je `key_hpss_cqt` bio tačan dok je `key_hpss` (STFT varijanta) bio netačan.

Zajedno, ove tri ćelije daju precizan, numerički uvid u to koliko konkretnih, pojedinačnih numera je CQT transformacija „spasila“ u odnosu na STFT, za svaku od tri metode izolacije signala posebno, što predstavlja znatno detaljniju i ubedljiviju potvrdu prednosti CQT-a od same razlike agregatnih procenata tačnosti prikazanih u odeljku 2.6.

## 2.10 Analiza po žanrovima (drugi skup podataka)

Kao i u prvom delu rada, važno je proveriti da li se ranije uočeni žanrovski obrasci (veći izazov kod harmonijski složenijih žanrova) i dalje javljaju nakon uvođenja CQT transformacije, kao i da li je poboljšanje ravnomerno raspoređeno po svim žanrovima ili je koncentrisano samo na pojedine.

#### Analiza za BPM

In [40]:
genre_bpm = df.groupby("genre")[
    [
        "bpm_error_song",
        "bpm_error_drums",
        "bpm_error_hpss"
    ]
].mean()

genre_bpm.sort_values("bpm_error_drums")

,bpm_error_song,bpm_error_drums,bpm_error_hpss
genre,,,
house,0.108,0.108,0.108
reggae,0.384,0.384,0.384
kpop,0.454,0.454,0.454
disco,0.874,0.874,0.874
funk,0.890,0.890,0.890
electronic,0.940,0.940,0.940
rock,0.946,0.946,0.946
hiphop,5.017,0.966,0.951
techno,1.030,1.030,1.030


#### Analiza za tonalitet

Koristimo kolonu *key_demucs_cqt* jer ima najbolji accuracy. Ova odluka proizilazi direktno iz rezultata prethodno prikazanih u odeljku 2.7, gde se pokazalo da kombinacija Demucs izolacije i CQT transformacije nadmašuje sve ostale razmatrane pristupe. Analiza po žanru se stoga fokusira na ovu, najbolju raspoloživu metodu.

In [41]:
genre_key = df.groupby("genre").apply(
    lambda x:
    (
        x["key_demucs_cqt"]
        ==
        x["manual_key"]
    ).mean() * 100
)

genre_key = genre_key.sort_values(
    ascending=False
)

genre_key

genre
acoustic      100.0
disco         100.0
folk          100.0
electronic    100.0
blues          80.0
reggae         80.0
rap            80.0
rock           80.0
country        60.0
jazz           60.0
soul           60.0
kpop           60.0
trance         60.0
techno         60.0
pop            40.0
metal          40.0
latin          40.0
house          40.0
funk           40.0
hiphop         40.0
rnb            40.0
punk           40.0
dtype: float64

Za razliku od metoda koje su koristile STFT, ova metoda nije ni za jedan žanr pogrešila u detekciji tonaliteta za svaku pesmu, čak šta više, za svaki žanr, ova metoda daje vrlo dobar rezultat. Ovo je snažna potvrda opšte primenljivosti CQT+Demucs pristupa, dok je u prvom delu rada uočeno da pojedini žanrovi (fank, džez, latino, soul) sistematski dovode do potpunog neuspeha detekcije, ovde takav ekstremni obrazac izostaje, što ukazuje da CQT transformacija ublažava upravo one nedostatke koji su bili najizraženiji kod harmonijski složenijih žanrova.

## 2.11 Pregled numera kod kojih CQT „pobeđuje“ STFT

U ovom odeljku detaljno se prikazuje **konkretan spisak numera** (a ne samo brojčani zbir, kao u odeljku 2.9) kod kojih je CQT metoda ispravno odredila tonalitet, dok je odgovarajuća STFT metoda pogrešila za sve tri metode izolacije signala. Ovakav detaljan, poimenični pregled omogućava dublju, kvalitativnu analizu; na primer, uočavanje da li „pobede“ CQT-a preovlađuju kod određenih žanrova ili specifičnih tonaliteta, što otvara prostor za dalje muzikološko tumačenje dobijenih rezultata.

#### Za celu pesmu

In [42]:
df[(df["key_song_cqt"] == df["manual_key"]) & (df["key_song"] != df["manual_key"])][
    [
        "song_name",
        "genre",
        "manual_key",
        "key_song",
        "key_song_cqt"
    ]
]

,song_name,genre,manual_key,key_song,key_song_cqt
2,Oasis - Wonderwall.wav,acoustic,F# Minor,E Major,F# Minor
3,The Beatles - Blackbird.wav,acoustic,G Major,G Minor,G Major
4,Tracy Chapman - Fast Car.wav,acoustic,A Major,A Minor,A Major
5,Albert King - Born Under A Bad Sign.wav,blues,C# Minor,C# Major,C# Minor
6,B.B. King - The Thrill Is Gone.wav,blues,B Minor,B Major,B Minor
7,Robert Johnson - Cross Road Blues.wav,blues,B Major,A# Minor,B Major
12,John Denver - Take Me Home Country Roads.wav,country,A Major,E Major,A Major
13,Johnny Cash - Ring of Fire.wav,country,G Major,D Major,G Major
16,Bee Gees - Stayin Alive.wav,disco,F Minor,F Major,F Minor
23,deadmau5 - Strobe.wav,electronic,G# Minor,B Major,G# Minor


#### Za Demucs

In [43]:
df[(df["key_demucs_cqt"] == df["manual_key"]) & (df["key_demucs"] != df["manual_key"])][
    [
        "song_name",
        "genre",
        "manual_key",
        "key_demucs",
        "key_demucs_cqt"
    ]
]

,song_name,genre,manual_key,key_demucs,key_demucs_cqt
2,Oasis - Wonderwall.wav,acoustic,F# Minor,E Major,F# Minor
3,The Beatles - Blackbird.wav,acoustic,G Major,G Minor,G Major
7,Robert Johnson - Cross Road Blues.wav,blues,B Major,B Minor,B Major
12,John Denver - Take Me Home Country Roads.wav,country,A Major,E Major,A Major
16,Bee Gees - Stayin Alive.wav,disco,F Minor,F Major,F Minor
23,deadmau5 - Strobe.wav,electronic,G# Minor,B Major,G# Minor
29,The Animals - House Of The Rising Sun.wav,folk,A Minor,E Major,A Minor
31,James Brown - Get Up Offa That Thing.wav,funk,E Minor,F# Minor,E Minor
42,Eric Prydz - Pjanoo.wav,house,G Minor,D Minor,G Minor
44,Swedish House Mafia ft. John Martin - Dont You...,house,B Minor,D Major,B Minor


#### Za HPSS

In [44]:
df[(df["key_hpss_cqt"] == df["manual_key"]) & (df["key_hpss"] != df["manual_key"])][
    [
        "song_name",
        "genre",
        "manual_key",
        "key_hpss",
        "key_hpss_cqt"
    ]
]

,song_name,genre,manual_key,key_hpss,key_hpss_cqt
3,The Beatles - Blackbird.wav,acoustic,G Major,G Minor,G Major
4,Tracy Chapman - Fast Car.wav,acoustic,A Major,A Minor,A Major
7,Robert Johnson - Cross Road Blues.wav,blues,B Major,B Minor,B Major
12,John Denver - Take Me Home Country Roads.wav,country,A Major,E Major,A Major
13,Johnny Cash - Ring of Fire.wav,country,G Major,D Major,G Major
16,Bee Gees - Stayin Alive.wav,disco,F Minor,F Major,F Minor
23,deadmau5 - Strobe.wav,electronic,G# Minor,B Major,G# Minor
25,Bob Dylan - Blowin in the Wind.wav,folk,D Major,A Major,D Major
34,Wild Cherry - Play That Funky.wav,funk,E Minor,E Major,E Minor
36,Kendrick Lamar - HUMBLE.wav,hiphop,C# Minor,E Minor,C# Minor


Ove tri tabele, razmatrane zajedno sa brojčanim rezimeom iz odeljka 2.9, predstavljaju najkonkretniji, najopipljiviji dokaz vrednosti uvođenja CQT transformacije u ovom radu.

## 2.12 Slučajevi gde greše sve raspoložive metode (STFT i CQT zajedno)

Na samom kraju analize, vraćamo se pitanju postavljenom i u prvom delu rada - koliko numera predstavlja **apsolutni, sistemski izazov** za sve raspoložive pristupe, ovoga puta uzimajući u obzir svih šest metoda (tri STFT + tri CQT) istovremeno, umesto samo tri STFT metode kao u odeljku 1.10. Ovo predstavlja najstrožiji mogući kriterijum neuspeha; numera mora biti pogrešno klasifikovana od strane **svake** pojedinačne metode koju je ovaj eksperiment razmatrao.

In [45]:
all_wrong = df[
    (df["key_song"] != df["manual_key"]) &
    (df["key_demucs"] != df["manual_key"]) &
    (df["key_hpss"] != df["manual_key"]) &
    (df["key_song_cqt"] != df["manual_key"]) &
    (df["key_demucs_cqt"] != df["manual_key"]) &
    (df["key_hpss_cqt"] != df["manual_key"])
]

print(f"Sve metode su pogrešile u {len(all_wrong)} slučaja.")

Sve metode su pogrešile u 18 slučaja.


Poređenjem ovog broja sa brojem od 41 numere iz prvog dela rada (gde je razmatrano samo tri metode), očekuje se, i eksperimentalno se potvrđuje, znatno manji broj „potpuno izgubljenih“ slučajeva, budući da dodavanje tri nove, po pravilu tačnije CQT metode povećava šansu da bar jedna od ukupno šest metoda „pogodi“ tačan tonalitet za datu numeru.

#### Provera po žanrovima gde su sve metode pogrešile

In [46]:
all_wrong["genre"].value_counts()

genre
funk       2
jazz       2
latin      2
soul       2
country    1
hiphop     1
house      1
kpop       1
metal      1
punk       1
reggae     1
rnb        1
rock       1
techno     1
Name: count, dtype: int64

Poređenjem sa analognim rezultatom iz odeljka 1.12 (prvi deo rada), moguće je proceniti da li su konkretni, ranije problematični žanrovi (fank, džez, latino, soul) i dalje disproporcionalno zastupljeni među preostalim, najtvrdokornijim slučajevima grešaka, čime se zaokružuje detaljna, uporedna analiza sprovedena u ovom radu.

---

# Zaključak

Ovaj tutorijal je, kroz analizu rezultata sprovedenih nad skupom od 110 muzičkih numera raspoređenih u širok spektar žanrova, ilustrovao ceo proces ekstrakcije i evaluacije dva fundamentalna muzička elementa iz audio signala, **tempa (BPM)** i **tonaliteta (key)**, kao i uticaj izbora tehnika obrade signala na kvalitet tih rezultata. Na osnovu sprovedene analize mogu se izvesti sledeći ključni zaključci:

**1. Detekcija tempa je robustan i pouzdan zadatak**, pod uslovom da se prilikom evaluacije uvaži fenomen oktavne (half-time/double-time) greške. Izolacija bubnjeva korišćenjem modela dubokog učenja (Demucs) donosi merljivo, ali ne dramatično poboljšanje u odnosu na klasičnu, signal-processing HPSS separaciju, dok je analiza tempa nad celom, nerazdvojenom pesmom znatno manje pouzdana usled međusobnog ometanja instrumenata u polifonoj mešavini.

**2. Detekcija tonaliteta je znatno teži i osetljiviji zadatak od detekcije tempa.** Klasičan pristup zasnovan na kratkotrajnoj Furijeovoj transformaciji (STFT) postiže tačnost od svega oko 53–54% čak i uz upotrebu izolacije instrumenata, dok u trećini slučajeva sve razmatrane STFT zasnovane metode zajedno greše. Ovaj nalaz potvrđuje da je izbor same transformacije (a ne samo izbor izolovanog signala nad kojim se ona primenjuje) kritičan faktor kvaliteta detekcije.

**3. Uvođenje transformacije sa konstantnim Q faktorom (CQT), čija logaritamska frekvencijska skala verno odslikava geometrijsku prirodu muzičke visine tona, donosi konzistentno i značajno poboljšanje tačnosti detekcije tonaliteta**; za svaku od tri razmatrane metode izolacije signala, CQT varijanta nadmašuje odgovarajuću STFT varijantu, sa najboljim rezultatom od 63.64% tačnosti kod kombinacije CQT transformacije i Demucs izolacije harmonijskih instrumenata. Kombinovanjem svih šest raspoloživih metoda (STFT i CQT), procenat numera kod kojih **sve** metode greše pada sa oko 35% na oko 16%, što predstavlja izuzetno vredno poboljšanje sa praktičnog stanovišta.

**4. Naivno glasanje između metoda, ponderisano merom pouzdanosti (confidence), nije se pokazalo kao uspešna ansambl strategija**, prevashodno zbog toga što dve od tri razmatrane STFT metode (analiza cele pesme i HPSS harmonijska komponenta) daju gotovo identične rezultate, čime efektivno „nadglašavaju“ treću, statistički tačniju Demucs metodu.

**5. Muzički žanr predstavlja značajan, sistematski faktor u tačnosti detekcije tonaliteta.** Žanrovi sa harmonijski složenijom strukturom, poput džeza, fanka, soul-a i latino muzike, koji obiluju proširenim akordima, modalnim interčejndžom i čestim modulacijama, sistematski predstavljaju najveći izazov za sve razmatrane pristupe, dok se ovaj problem delom, ali ne u potpunosti, ublažava uvođenjem CQT transformacije.